# Embedding Types in Machine Learning: Code + Documentation

This notebook is a practical guide to the **major embedding types** used in modern ML systems.

It includes:
- concise theory
- commented code
- small runnable examples

## What you will cover
1. Categorical embeddings (learned lookup tables)
2. Word embeddings (Word2Vec, FastText style)
3. Contextual embeddings (Transformer/BERT style)
4. Sentence and document embeddings
5. Image embeddings
6. Graph/node embeddings
7. User-item embeddings (recommender systems)

> Note: 
 in practice is very broad. This notebook focuses on the core families used most often in production and research.

## 0) Setup
The code below imports common libraries and creates utility helpers.

In [ ]:
import math
import random
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# Optional imports (kept in try blocks later):
# tensorflow, gensim, sentence_transformers, networkx

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def show_similarities(vectors, labels, top_k=3):
    """Print top-k cosine similarities for each vector."""
    sims = cosine_similarity(vectors)
    for i, label in enumerate(labels):
        order = np.argsort(-sims[i])
        neighbors = [(labels[j], float(sims[i, j])) for j in order if j != i][:top_k]
        print(f"{label:>12} -> {neighbors}")

In [ ]:
import importlib.util
import os


def load_runtime_env() -> None:
    candidate_paths = [
        os.path.join(os.getcwd(), "configs", "runtime.env"),
        os.path.join(os.getcwd(), "configs", "runtime.env.example"),
        os.path.join(os.getcwd(), "..", "configs", "runtime.env"),
        os.path.join(os.getcwd(), "..", "configs", "runtime.env.example"),
    ]
    for path in candidate_paths:
        if not os.path.exists(path):
            continue
        with open(path, "r", encoding="utf-8") as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip("\"'")
                if key and key not in os.environ:
                    os.environ[key] = value
        break


def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}


load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))
NO_CUDA = not USE_GPU

_torch_cuda = False
_tf_gpu = False
if importlib.util.find_spec("torch") is not None:
    import torch

    _torch_cuda = torch.cuda.is_available()
if importlib.util.find_spec("tensorflow") is not None:
    import tensorflow as tf

    _tf_gpu = bool(tf.config.list_physical_devices("GPU"))

RUNTIME_DEVICE = "cuda" if USE_GPU and (_torch_cuda or _tf_gpu) else "cpu"
print(f"USE_GPU={int(USE_GPU)} | runtime_device={RUNTIME_DEVICE}")

## 1) Categorical Embeddings (Learned Lookup Tables)

A categorical embedding maps each discrete ID (for example: city, product, user) to a dense vector.

Mathematically, for category index $i$, embedding is $e_i n athbb{R}^d$, learned by gradient descent with the task loss.

In [ ]:
# Tiny example: learn embeddings for 10 categories in a binary task.
# If TensorFlow is unavailable, this cell prints a fallback note.

try:
    import tensorflow as tf
    from tensorflow import keras

    tf.random.set_seed(SEED)

    n_categories = 10
    embed_dim = 4

    # Synthetic data: category ids and target label pattern.
    x = np.random.randint(0, n_categories, size=(2000, 1))
    y = ((x[:, 0] % 2) == 0).astype(np.float32)  # even ids -> class 1

    inp = keras.Input(shape=(1,), dtype="int32")
    emb = keras.layers.Embedding(input_dim=n_categories, output_dim=embed_dim, name="cat_embedding")(inp)
    flat = keras.layers.Flatten()(emb)
    out = keras.layers.Dense(1, activation="sigmoid")(flat)

    model = keras.Model(inp, out)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(x, y, epochs=5, batch_size=32, verbose=0)

    learned = model.get_layer("cat_embedding").get_weights()[0]
    print("Embedding matrix shape:", learned.shape)
    print("First 3 vectors:
", learned[:3])
except Exception as e:
    print("TensorFlow not available for this section.")
    print("Install tensorflow to run this cell. Error:", e)

## 2) Word Embeddings

Word embeddings encode words into dense vectors so semantic neighbors are close in cosine space.

### 2.1 Word2Vec-style (static embeddings)
Word gets one vector regardless of context.

In [ ]:
sentences = [
    "machine learning improves search relevance".split(),
    "deep learning enables representation learning".split(),
    "embeddings capture semantic similarity".split(),
    "neural networks learn dense vectors".split(),
    "search ranking uses vector similarity".split(),
]

try:
    from gensim.models import Word2Vec

    w2v = Word2Vec(sentences=sentences, vector_size=32, window=3, min_count=1, workers=1, seed=SEED)
    words = ["learning", "search", "similarity", "vectors"]
    vecs = np.vstack([w2v.wv[w] for w in words])
    show_similarities(vecs, words)
except Exception as e:
    print("gensim not available for Word2Vec.")
    print("Install gensim to run this exact example. Error:", e)

### 2.2 FastText-style (subword-aware static embeddings)
FastText uses character n-grams, helping with rare and misspelled words.

In [ ]:
try:
    from gensim.models import FastText

    ft = FastText(sentences=sentences, vector_size=32, window=3, min_count=1, workers=1, seed=SEED)
    words = ["learning", "learner", "vector", "vectors"]
    vecs = np.vstack([ft.wv[w] for w in words])
    show_similarities(vecs, words)
except Exception as e:
    print("gensim not available for FastText.")
    print("Install gensim to run this exact example. Error:", e)

## 3) Contextual Embeddings (Transformer/BERT style)

Contextual embeddings produce **different vectors for the same word** depending on surrounding text.

Example idea:
- "bank" in "river bank"
- "bank" in "open a bank account"

These should have different vectors with contextual models.

In [ ]:
texts = [
    "The fisherman sat near the river bank.",
    "She went to the bank to open a savings account.",
]

try:
    from sentence_transformers import SentenceTransformer

    # Lightweight sentence-level contextual embeddings.
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    emb = encoder.encode(texts, normalize_embeddings=True)

    print("Embedding shape:", emb.shape)
    print("Cosine similarity between two contexts:", float(np.dot(emb[0], emb[1])))
except Exception as e:
    print("sentence-transformers not available.")
    print("Install sentence-transformers to run this section. Error:", e)

## 4) Sentence / Document Embeddings

You can represent whole texts as vectors for retrieval, clustering, classification, and semantic search.

Below is a classic baseline: TF-IDF + SVD (also called Latent Semantic Analysis).

In [ ]:
docs = [
    "Embeddings are useful for semantic search",
    "Neural networks learn dense representations",
    "Vector databases store embedding vectors",
    "Classical NLP uses tf-idf and topic models",
]

vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(docs)

# Reduce sparse TF-IDF to dense semantic space.
svd = TruncatedSVD(n_components=min(4, X_tfidf.shape[1] - 1), random_state=SEED)
X_dense = svd.fit_transform(X_tfidf)

print("Dense doc embedding shape:", X_dense.shape)
show_similarities(X_dense, [f"doc_{i}" for i in range(len(docs))])

## 5) Image Embeddings

Image embeddings are usually feature vectors from a pretrained CNN/ViT backbone.

Typical usage:
- remove classifier head
- use pooled penultimate features as embedding

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras

    # Build a feature extractor from MobileNetV2.
    backbone = keras.applications.MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

    # Dummy image batch for shape demonstration (replace with real preprocessed images).
    imgs = np.random.rand(2, 224, 224, 3).astype("float32") * 255.0
    imgs = keras.applications.mobilenet_v2.preprocess_input(imgs)

    img_emb = backbone.predict(imgs, verbose=0)
    print("Image embedding shape:", img_emb.shape)
except Exception as e:
    print("TensorFlow/Keras not available for image embeddings.")
    print("Install tensorflow to run this section. Error:", e)

## 6) Graph / Node Embeddings

Graph embeddings map nodes (or whole graphs) into vectors preserving topology/proximity.

Below is a simple spectral embedding baseline using graph adjacency.

In [ ]:
# Simple graph spectral embedding with NumPy only.
# For larger tasks, methods like node2vec, DeepWalk, GraphSAGE are common.

adj = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 0, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 0, 1, 0],
], dtype=float)

# Graph Laplacian L = D - A
deg = np.diag(adj.sum(axis=1))
lap = deg - adj

# Smallest non-trivial eigenvectors give node coordinates.
eigvals, eigvecs = np.linalg.eigh(lap)
node_emb = eigvecs[:, 1:3]  # 2D embedding

print("Node embedding shape:", node_emb.shape)
print(node_emb)

## 7) User-Item Embeddings (Recommender Systems)

Matrix-factorization style recommenders learn one embedding for each user and each item.
A rating or click score is often modeled via dot product:
$$
at{r}_{u,i} = e_u^T e_i
$$

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras

    tf.random.set_seed(SEED)

    n_users, n_items, d = 20, 30, 8
    n_samples = 3000

    # Synthetic implicit feedback (1 if user/item parity pattern matches, else 0).
    users = np.random.randint(0, n_users, size=(n_samples, 1))
    items = np.random.randint(0, n_items, size=(n_samples, 1))
    labels = ((users[:, 0] % 2) == (items[:, 0] % 2)).astype(np.float32)

    u_in = keras.Input(shape=(1,), dtype="int32", name="user")
    i_in = keras.Input(shape=(1,), dtype="int32", name="item")

    u_emb = keras.layers.Embedding(n_users, d, name="user_embedding")(u_in)
    i_emb = keras.layers.Embedding(n_items, d, name="item_embedding")(i_in)

    # Dot product similarity score between user/item vectors.
    score = keras.layers.Dot(axes=-1)([u_emb, i_emb])
    score = keras.layers.Flatten()(score)
    out = keras.layers.Activation("sigmoid")(score)

    rec_model = keras.Model([u_in, i_in], out)
    rec_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    rec_model.fit([users, items], labels, epochs=4, batch_size=64, verbose=0)

    U = rec_model.get_layer("user_embedding").get_weights()[0]
    I = rec_model.get_layer("item_embedding").get_weights()[0]
    print("User embedding matrix:", U.shape)
    print("Item embedding matrix:", I.shape)
except Exception as e:
    print("TensorFlow not available for recommender embeddings.")
    print("Install tensorflow to run this section. Error:", e)

## 8) Practical Selection Guide

Choose embedding type by data modality and task:
- Tabular categorical features: learned categorical embeddings
- Word-level NLP (lightweight): Word2Vec / FastText
- Modern NLP semantics: contextual Transformer embeddings
- Retrieval/search over texts: sentence/document embeddings
- Vision retrieval/similarity: image embeddings from pretrained backbones
- Graph analytics/link prediction: graph/node embeddings
- Recommenders: user-item dual embeddings

## 9) References
- Mikolov et al., Word2Vec
- Bojanowski et al., FastText
- Devlin et al., BERT
- Kipf and Welling, GCN
- He et al., Neural Collaborative Filtering